In [3]:
import subprocess
import re
from pathlib import Path
import sys
import tarfile # Importa a biblioteca para manipular arquivos .tar.gz
import os # Importa a biblioteca os para manipulação de caminhos

# Caminho raiz do projeto (onde está o pyproject.toml)
PROJECT_DIR = Path("./codebench-analytics-full").resolve()

# O diretório de extraídos está um nível acima do local atual.
EXTRAIDOS_DIR = Path("../Extraidos").resolve()


def extract_tar_gz_files(directory):
    """
    Procura por arquivos .tar.gz em um diretório e os extrai.
    Remove o arquivo .tar.gz após a extração bem-sucedida.
    """
    for item in os.listdir(directory):
        if item.endswith(".tar.gz"):
            tar_path = os.path.join(directory, item)
            print(f"📦 Encontrado arquivo compactado: {tar_path}. Extraindo...")
            try:
                with tarfile.open(tar_path, "r:gz") as tar:
                    # Extrai os arquivos para o mesmo diretório
                    tar.extractall(path=directory)
                print(f"✅ Extração de {item} concluída.")
                # Remove o arquivo .tar.gz após a extração
                os.remove(tar_path)
                print(f"🗑️ Arquivo {item} removido.")
            except Exception as e:
                print(f"❌ Erro ao extrair {tar_path}: {e}")


def extract_year_semester_from_folder(folder_name):
    """
    Extrai ano e semestre do nome da pasta.
    Exemplo: cb_dataset_2018_1_v1.81 -> (2018, 1)
    """
    match = re.search(r'cb_dataset_(\d{4})_(\d+)_v', folder_name)
    if match:
        year = match.group(1)
        semester = match.group(2)
        return year, semester
    return None, None

def discover_datasets():
    """
    Descobre, descompacta e prepara os caminhos dos datasets.
    """
    print(f"Procurando datasets em: {EXTRAIDOS_DIR}")
    
    if not EXTRAIDOS_DIR.exists():
        print(f"❌ Diretório Extraidos não encontrado: {EXTRAIDOS_DIR}")
        sys.exit(1)
    
    dataset_paths = []
    
    # Procurar por pastas que seguem o padrão cb_dataset_*
    for item in EXTRAIDOS_DIR.iterdir():
        if item.is_dir() and item.name.startswith("cb_dataset_"):
            year, semester = extract_year_semester_from_folder(item.name)
            
            if year and semester:
                # Construir o caminho para a pasta interna
                dataset_path = item / f"{year}-{semester}"
                
                if dataset_path.exists():
                    # --- NOVA ETAPA: VERIFICAR E EXTRAIR ARQUIVOS .TAR.GZ ---
                    extract_tar_gz_files(dataset_path)
                    
                    dataset_paths.append(str(dataset_path.resolve()))
                    print(f"✅ Dataset preparado: {item.name} -> {dataset_path}")
                else:
                    print(f"⚠️  Pasta interna não encontrada: {dataset_path}")
            else:
                print(f"⚠️  Não foi possível extrair ano/semestre de: {item.name}")
    
    if not dataset_paths:
        print("❌ Nenhum dataset válido encontrado!")
        sys.exit(1)
    
    # Ordenar os caminhos para processamento consistente
    dataset_paths.sort()
    
    print(f"\n📊 Total de datasets preparados: {len(dataset_paths)}")
    return dataset_paths

dataset_paths = discover_datasets()




Procurando datasets em: /root/QuestionInsight/Extraidos
✅ Dataset preparado: cb_dataset_2024_1_v1.81 -> /root/QuestionInsight/Extraidos/cb_dataset_2024_1_v1.81/2024-1

📊 Total de datasets preparados: 1


In [4]:
import subprocess
from pathlib import Path
import sys

# Caminho raiz do projeto (onde está o pyproject.toml)
#PROJECT_DIR = Path("/home/supremo/Downloads/Trabalho_CodeBench_WEI/Etapa_4/codebench-analytics-full").resolve()

def run_makefile():
    print(f"Executando Makefile em: {PROJECT_DIR}")
    result = subprocess.run(["make"], cwd=PROJECT_DIR)
    if result.returncode != 0:
        print("❌ Erro ao executar o Makefile.")
        sys.exit(1)

def run_execution_command(*paths):
    print("Executando análise de execução para os seguintes datasets:")
    for p in paths:
        print(f" - {p}")
    
    # Cria uma lista com um '-p' antes de cada caminho
    p_args = []
    for p in paths:
        p_args.extend(["-p", p])
    
    command = ["poetry", "run", "codebench_analytics", "execution", *p_args, "-k", "exam"]
    result = subprocess.run(command, cwd=PROJECT_DIR)
    if result.returncode != 0:
        print("❌ Erro ao executar o comando de execução.")
        sys.exit(1)

def run_action_command(*paths):
    print("Executando análise de ações para os seguintes datasets:")
    for p in paths:
        print(f" - {p}")
    
    # Cria uma lista com um '-p' antes de cada caminho
    p_args = []
    for p in paths:
        p_args.extend(["-p", p])
    
    command = ["poetry", "run", "codebench_analytics", "action", *p_args, "-k", "exam"]
    result = subprocess.run(command, cwd=PROJECT_DIR)
    if result.returncode != 0:
        print("❌ Erro ao executar o comando de ação.")
        sys.exit(1)

def run_solution_command(solution_path):
    print(f"Extraindo métricas da solução: {solution_path}")
    command = ["poetry", "run", "codebench_analytics", "solution", "-p", solution_path]
    result = subprocess.run(command, cwd=PROJECT_DIR)
    if result.returncode != 0:
        print("❌ Erro ao extrair métricas da solução.")
        sys.exit(1)

def main():
    run_makefile()
    
    # Lista dos datasets — altere conforme necessário
    """dataset_paths = [
        "/home/supremo/Downloads/Trabalho_CodeBench_WEI/Extraidos/cb_dataset_2018_1_v1.81/2018-1",
        "/home/supremo/Downloads/Trabalho_CodeBench_WEI/Extraidos/cb_dataset_2018_2_v1.81/2018-2",
    ]
    
    # Resolve para caminhos absolutos
    dataset_paths = [str(Path(p).resolve()) for p in dataset_paths]
    """
    # Executa tanto execution quanto action
    run_execution_command(*dataset_paths)
    run_action_command(*dataset_paths)
    
    # Caminho para o CSV da solução
    solution_csv = PROJECT_DIR / "codigo_solucao.csv"
    if not solution_csv.exists():
        print(f"❌ Arquivo de solução não encontrado: {solution_csv}")
        sys.exit(1)
    
    run_solution_command(str(solution_csv))

if __name__ == "__main__":
    main()

Executando Makefile em: /root/QuestionInsight/Etapa_4/codebench-analytics-full
⚠️  Poetry não encontrado. Tentando instalar via apt...
/usr/bin/poetry
Hit:1 http://archive.ubuntu.com/ubuntu noble InRelease
Get:2 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:3 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Hit:4 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:6 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:7 http://security.ubuntu.com/ubuntu noble-security/main amd64 Components [21.6 kB]
Get:8 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Components [52.2 kB]
Get:9 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1335 kB]
Get:10 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Components [212 B]
Get:11 http://security.ubuntu.com/ubuntu noble-secur

All done! ✨ 🍰 ✨
19 files would be left unchanged.


All checks passed!
Executando análise de execução para os seguintes datasets:
 - /root/QuestionInsight/Extraidos/cb_dataset_2024_1_v1.81/2024-1
2025-08-18 15:08:11,259 codebench_analytics.extractor.execution_extractor INFO:extracting execution data from '/root/QuestionInsight/Extraidos/cb_dataset_2024_1_v1.81/2024-1'
2025-08-18 15:08:32,272 root INFO:generating 'executions_data.csv' file into 'output/data/executions_data.csv'
2025-08-18 15:08:32,427 codebench_analytics.collector.executions INFO:collecting metrics from 'output/data/executions_data.csv'
2025-08-18 15:08:32,583 root INFO:generating 'executions_by_student.csv' file into 'output/metrics/executions_by_student.csv'
2025-08-18 15:08:32,594 root INFO:generating 'executions.csv' file into 'output/metrics/executions.csv'
Executando análise de ações para os seguintes datasets:
 - /root/QuestionInsight/Extraidos/cb_dataset_2024_1_v1.81/2024-1
2025-08-18 15:08:33,969 codebench_analytics.extractor.action_extractor INFO:extracting 'co

# Explicação do Código

## O que o código faz:
1. **Carregamento de Arquivos CSV**:
   - Carrega dois arquivos CSV usando `pandas`:
     - `actions_data.csv`: contém informações sobre tempos e eventos relacionados a ações.
     - `executions.csv`: contém informações sobre interações e resultados relacionados às execuções.

2. **Renomeação de Colunas**:
   - Renomeia algumas colunas dos dois arquivos para facilitar a combinação e tornar os nomes mais descritivos:
     - No `actions_data.csv`:
       - `code_time` → `tempo_implementacao`
       - `num_events` → `num_eventos`
       - `num_deletes` → `num_eventos_del`
     - No `executions.csv`:
       - `num_submissions` → `num_submissoes`
       - `num_students_interactions` → `num_consultas`
       - `amount_of_change` → `qtd_alteracoes_codigo`

3. **União dos Dados**:
   - Combina os dois DataFrames com base na coluna `question`, utilizando um merge à esquerda (`how='left'`), o que preserva todas as linhas de `executions.csv`.

4. **Seleção de Colunas**:
   - Cria um novo DataFrame contendo apenas as colunas relevantes para a análise:
     - `question`, `tempo_implementacao`, `num_eventos`, `num_eventos_del`, `num_consultas`, `num_submissoes`, `num_tests`, `num_correct`, `num_errors`, `num_logic_errors`, `num_syntax_errors`, `qtd_alteracoes_codigo`.

5. **Exportação para CSV**:
   - Salva o DataFrame resultante em um novo arquivo chamado `resultado.csv`.

6. **Mensagem de Sucesso**:
   - Exibe uma mensagem no console indicando que o arquivo foi gerado com sucesso.

## Resultado:
- Um arquivo CSV chamado `resultado.csv`, contendo as colunas combinadas e selecionadas dos dois arquivos originais, está pronto para ser usado em análises ou relatórios.


In [5]:
import pandas as pd

# Carregar os dois arquivos CSV
df1 = pd.read_csv(r'codebench-analytics-full/output/metrics/actions_data.csv')  # Primeiro arquivo com os tempos e eventos
df2 = pd.read_csv(r'codebench-analytics-full/output/metrics/executions.csv')  # Segundo arquivo com as interações e resultados

# Renomear colunas para facilitar a combinação
df1.rename(columns={
    'code_time': 'tempo_implementacao',
    'num_events': 'num_eventos',
    'num_deletes': 'num_eventos_del'
}, inplace=True)

# Renomear colunas do segundo DataFrame
df2.rename(columns={
    'num_submissions': 'num_submissoes',
    'num_students_interactions': 'num_consultas',  # Renomeando para compatibilidade
    'amount_of_change': 'qtd_alteracoes_codigo'  # Renomeando para compatibilidade
}, inplace=True)

# Unir os DataFrames com base na coluna 'question'
merged_df = pd.merge(df2, df1, on='question', how='left')

# Selecionar apenas as colunas desejadas para o novo DataFrame
final_df = merged_df[['question', 'tempo_implementacao', 'num_eventos', 'num_eventos_del', 
                       'num_consultas', 'num_submissoes',  # Usando 'num_consultas' renomeado
                       'num_tests', 'num_correct', 'num_errors', 
                       'num_logic_errors', 'num_syntax_errors', 'qtd_alteracoes_codigo']]  # Usando 'qtd_alteracoes_codigo' renomeado

# Salvar o DataFrame resultante em um novo arquivo CSV
final_df.to_csv('resultado.csv', index=False)

print("Arquivo 'resultado.csv' gerado com sucesso!")


Arquivo 'resultado.csv' gerado com sucesso!


## Explicação do Código

Este código tem como objetivo ler um arquivo CSV, manipular os dados e salvar um novo arquivo com os resultados. O processo é o seguinte:

1. **Leitura do arquivo CSV**: 
   O arquivo `'unified_solutions.csv'` é carregado em um DataFrame utilizando a biblioteca `pandas`.

2. **Criação de nova coluna**:
   Uma nova coluna chamada `'id'` é criada, copiando os valores da coluna `'assignment'`.

3. **Remoção de duplicatas**:
   A coluna `'id'` é filtrada para remover duplicatas, criando um novo DataFrame com valores únicos.

4. **Salvamento dos dados**:
   O novo DataFrame é salvo em um arquivo CSV chamado `'assessments.csv'`.

5. **Mensagem de sucesso**:
   Uma mensagem é exibida para informar que o arquivo de saída foi criado com sucesso.

### Código

```python
import pandas as pd

input_file = 'unified_solutions.csv'
output_file = 'assessments.csv'

df = pd.read_csv(input_file)
df['id'] = df['assignment']
df_output = df[['id']].drop_duplicates()

df_output.to_csv(output_file, index=False)

print(f"Arquivo '{output_file}' criado com sucesso!")


In [6]:
import pandas as pd

# Caminho do arquivo de entrada e nome do arquivo de saída
input_file = '../CSVS_JO/unified_solutions.csv'  # Substitua pelo nome do seu arquivo CSV
output_file = 'assessments.csv'

# Carrega o CSV de entrada
df = pd.read_csv(input_file)

# Cria a coluna 'id' a partir da coluna 'assignment'
df['id'] = df['assignment']

# Remove duplicatas da coluna 'id'
df_output = df[['id']].drop_duplicates()

# Salva o novo DataFrame em um arquivo CSV
df_output.to_csv(output_file, index=False)

print(f"Arquivo '{output_file}' criado com sucesso!")


Arquivo 'assessments.csv' criado com sucesso!


# Explicação do Código

## O que o código faz:

1. **Importação do Módulo CSV**:
   - Utiliza o módulo `csv` do Python para manipular arquivos CSV.

2. **Função para Leitura de CSV**:
   - A função `ler_csv`:
     - Recebe o nome de um arquivo CSV.
     - Lê o conteúdo do arquivo e retorna uma lista de dicionários, onde cada linha do CSV é representada como um dicionário.

3. **Leitura dos Arquivos de Entrada**:
   - `dados_question`: Contém informações sobre as questões (ex.: tempo de implementação, número de eventos, etc.), lido de `resultado.csv`.
   - `dados_respostas`: Contém informações adicionais sobre as questões (ex.: dificuldade, discriminação, etc.), lido de `questoes_ordenadas.csv`.

4. **Combinação dos Dados**:
   - Para cada questão em `dados_question`:
     - Procura a questão correspondente em `dados_respostas` (compara a coluna `id` com `question`).
     - Cria um novo registro com informações combinadas de ambos os arquivos.

5. **Estrutura do Novo Registro**:
   - As informações do novo registro incluem:
     - Dados da questão (`tempo_implementacao`, `num_eventos`, `num_tests`, etc.).
     - Dados adicionais das respostas (`dificuldade`, `discriminacao`, etc.).

6. **Escrita do Arquivo Resultante**:
   - Salva os registros combinados em um novo arquivo chamado `question_new_info.csv`:
     - Escreve um cabeçalho com os nomes das colunas.
     - Adiciona as linhas dos registros combinados.

7. **Mensagem de Sucesso**:
   - Exibe uma mensagem no console indicando que os dados foram combinados e salvos com sucesso.

## Resultado:
- Um arquivo CSV chamado `question_new_info.csv` é gerado, contendo informações combinadas de `resultado.csv` e `questoes_ordenadas.csv`.
- Esse arquivo pode ser usado para análises completas das questões e suas respectivas métricas.


In [7]:
import csv

# Função para ler um CSV e retornar como uma lista de dicionários
def ler_csv(nome_arquivo):
    with open(nome_arquivo, mode='r', newline='') as arquivo_csv:
        leitor = csv.DictReader(arquivo_csv)
        return [linha for linha in leitor]

# Lendo os arquivos CSV de entrada
dados_question = ler_csv(r'resultado.csv')
dados_respostas = ler_csv(r'../Etapa_3/questoes_ordenadas.csv')

# Criar um novo registro para cada linha de dados_question
resultados = []

for question in dados_question:
    # Encontrar a resposta correspondente
    for resposta in dados_respostas:
        if int(resposta['id']) == int(question['question']):
            novo_registro = {
                "question": question['question'],
                "tempo_implementacao": question['tempo_implementacao'],
                "num_eventos": question['num_eventos'],
                "num_eventos_del": question['num_eventos_del'],
                "num_consultas": question['num_consultas'],
                "num_submissoes": question['num_submissoes'],
                "num_tests": question['num_tests'],
                "num_correct": question['num_correct'],
                "num_errors": question['num_errors'],
                "num_logic_errors": question['num_logic_errors'],
                "num_syntax_errors": question['num_syntax_errors'],
                "qtd_alteracoes_codigo": question['qtd_alteracoes_codigo'],
                "dificuldade": resposta['dificuldade'],
                "discriminacao": resposta['discriminacao'],
                "respostas": resposta['respostas'],
                "usuarios_respondidos": resposta['usuarios_respondidos']
            }
            resultados.append(novo_registro)

# Escrever os resultados em um novo arquivo CSV
with open('question_new_info.csv', mode='w', newline='') as arquivo_csv:
    campos = resultados[0].keys()
    escritor_csv = csv.DictWriter(arquivo_csv, fieldnames=campos)

    escritor_csv.writeheader()
    escritor_csv.writerows(resultados)

print("Dados combinados e salvos em question_new_info.csv")


Dados combinados e salvos em question_new_info.csv


In [8]:
import pandas as pd

def ordenar_question(arquivo_question):
    # Ler o arquivo CSV
    question_df = pd.read_csv(arquivo_question)

    # Ordenar o DataFrame pela coluna 'question'
    question_df_sorted = question_df.sort_values(by='question', ascending=True)

    # Sobrescrever o arquivo original com o DataFrame ordenado
    question_df_sorted.to_csv(arquivo_question, index=False)

    print(f"Arquivo '{arquivo_question}' foi ordenado e salvo com sucesso.")

# Exemplo de uso
arquivo_question = 'question_new_info.csv'
ordenar_question(arquivo_question)


Arquivo 'question_new_info.csv' foi ordenado e salvo com sucesso.


In [9]:
import pandas as pd

def comparar_e_gerar_csv(arquivo_question, arquivo_metrics, arquivo_saida):
    # Ler os arquivos CSV
    question_df = pd.read_csv(arquivo_question)
    metrics_df = pd.read_csv(arquivo_metrics)

    # Renomear a coluna 'question' para 'question_id' em question_df para facilitar a junção
    question_df.rename(columns={'question': 'question_id'}, inplace=True)

    # Fazer a junção dos DataFrames com base na coluna 'question_id'
    merged_df = pd.merge(metrics_df, question_df, on='question_id', how='inner')

    # Ordenar o DataFrame resultante com base na ordem de 'question_id' do question_df original
    merged_df = merged_df.sort_values(by='question_id')

    # Selecionar as colunas que você deseja manter
    colunas_a_manter = metrics_df.columns.tolist()  # Manter todas as colunas do metrics_df

    # Salvar o DataFrame resultante em um novo arquivo CSV
    merged_df.to_csv(arquivo_saida, columns=colunas_a_manter, index=False)

# Exemplo de uso com uma string "raw"
comparar_e_gerar_csv(
    'question_new_info.csv', 
    r'codebench-analytics-full/output/data/code_metrics_professor.csv', #Etapa_4/codebench-analytics-full/output
    'code_metrics_professor_util.csv'
)

print("Arquivo gerado: code_metrics_professor_util.csv")



# Ler o arquivo CSV
df = pd.read_csv('code_metrics_professor_util.csv')

# Renomear a coluna 'question_id' para 'question'
df.rename(columns={'question_id': 'question'}, inplace=True)

# Salvar o DataFrame com o novo nome da coluna
df.to_csv('code_metrics_professor_util.csv', index=False)

print("Nome da coluna alterado com sucesso no arquivo 'code_metrics_professor_util.csv'.")


Arquivo gerado: code_metrics_professor_util.csv
Nome da coluna alterado com sucesso no arquivo 'code_metrics_professor_util.csv'.
